# Module 4: Sequence Models

This notebook covers RNNs, LSTMs, and GRUs.

**Topics covered:**
- RNN fundamentals
- Vanishing gradients
- LSTM architecture
- Sequence-to-sequence models

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
%matplotlib inline

## 4.1 Vanilla RNN

In [ ]:
class VanillaRNN:
    """Simple RNN implementation."""
    
    def __init__(self, input_size, hidden_size, output_size):
        scale = 0.01
        self.Wxh = np.random.randn(hidden_size, input_size) * scale
        self.Whh = np.random.randn(hidden_size, hidden_size) * scale
        self.Why = np.random.randn(output_size, hidden_size) * scale
        self.bh = np.zeros((hidden_size, 1))
        self.by = np.zeros((output_size, 1))
        self.hidden_size = hidden_size
    
    def forward(self, inputs, h_prev):
        """
        Forward pass through sequence.
        
        Args:
            inputs: List of input vectors (each is (input_size, 1))
            h_prev: Initial hidden state (hidden_size, 1)
        """
        hs = {-1: h_prev}
        ys = []
        
        for t, x in enumerate(inputs):
            # Hidden state
            hs[t] = np.tanh(self.Wxh @ x + self.Whh @ hs[t-1] + self.bh)
            # Output
            y = self.Why @ hs[t] + self.by
            ys.append(y)
        
        return ys, hs
    
    def init_hidden(self):
        return np.zeros((self.hidden_size, 1))

In [ ]:
# Test RNN
rnn = VanillaRNN(input_size=10, hidden_size=32, output_size=10)
h = rnn.init_hidden()

# Create sequence of 5 inputs
inputs = [np.random.randn(10, 1) for _ in range(5)]
outputs, hiddens = rnn.forward(inputs, h)

print(f"Input sequence length: {len(inputs)}")
print(f"Output sequence length: {len(outputs)}")
print(f"Hidden state shape: {hiddens[0].shape}")

## 4.2 Vanishing Gradients Visualization

In [ ]:
def visualize_gradient_flow(seq_lengths=[5, 10, 20, 50]):
    """
    Visualize how gradients diminish over time steps.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Simulate gradient flow with different Whh eigenvalues
    eigenvalues = [0.9, 0.99, 1.0, 1.01]
    
    ax1 = axes[0]
    for ev in eigenvalues:
        T = 100
        gradient_magnitude = [ev ** t for t in range(T)]
        ax1.plot(gradient_magnitude, label=f'eigenvalue = {ev}')
    
    ax1.set_xlabel('Time steps back')
    ax1.set_ylabel('Gradient magnitude')
    ax1.set_title('Gradient Flow vs Whh Eigenvalue')
    ax1.legend()
    ax1.set_yscale('log')
    ax1.grid(True, alpha=0.3)
    
    # Show tanh saturation
    ax2 = axes[1]
    x = np.linspace(-5, 5, 100)
    tanh = np.tanh(x)
    tanh_deriv = 1 - tanh ** 2
    
    ax2.plot(x, tanh, 'b-', label='tanh(x)', linewidth=2)
    ax2.plot(x, tanh_deriv, 'r--', label="tanh'(x)", linewidth=2)
    ax2.axhline(y=0, color='k', linewidth=0.5)
    ax2.axhline(y=1, color='gray', linewidth=0.5, linestyle=':')
    ax2.set_xlabel('x')
    ax2.set_ylabel('Value')
    ax2.set_title('Tanh and its Derivative')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

visualize_gradient_flow()

## 4.3 LSTM Implementation

In [ ]:
class LSTMCell:
    """Single LSTM cell."""
    
    def __init__(self, input_size, hidden_size):
        self.hidden_size = hidden_size
        n = input_size + hidden_size
        
        # Concatenated weights for all gates
        scale = np.sqrt(2.0 / n)
        self.W = np.random.randn(4 * hidden_size, n) * scale
        self.b = np.zeros(4 * hidden_size)
        
        # Forget gate bias initialized to 1 (helps learning)
        self.b[hidden_size:2*hidden_size] = 1.0
    
    def forward(self, x, h_prev, c_prev):
        """
        Single LSTM step.
        
        Args:
            x: Input (input_size,)
            h_prev: Previous hidden state (hidden_size,)
            c_prev: Previous cell state (hidden_size,)
        
        Returns:
            h: New hidden state
            c: New cell state
        """
        H = self.hidden_size
        
        # Concatenate input and hidden
        combined = np.concatenate([x, h_prev])
        
        # All gates in one matmul
        gates = self.W @ combined + self.b
        
        # Split into individual gates
        i = self._sigmoid(gates[0:H])       # Input gate
        f = self._sigmoid(gates[H:2*H])     # Forget gate
        o = self._sigmoid(gates[2*H:3*H])   # Output gate
        g = np.tanh(gates[3*H:4*H])         # Candidate cell state
        
        # Update cell state
        c = f * c_prev + i * g
        
        # Update hidden state
        h = o * np.tanh(c)
        
        return h, c, {'i': i, 'f': f, 'o': o, 'g': g}
    
    def _sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

In [ ]:
# Test LSTM cell
lstm = LSTMCell(input_size=10, hidden_size=32)
x = np.random.randn(10)
h = np.zeros(32)
c = np.zeros(32)

h_new, c_new, gates = lstm.forward(x, h, c)

print(f"Input shape: {x.shape}")
print(f"Hidden state shape: {h_new.shape}")
print(f"Cell state shape: {c_new.shape}")
print(f"\nGate activations:")
for name, gate in gates.items():
    print(f"  {name}: mean={gate.mean():.3f}, std={gate.std():.3f}")

In [ ]:
# Visualize LSTM gates over a sequence
def visualize_lstm_gates(seq_len=50):
    lstm = LSTMCell(input_size=10, hidden_size=32)
    
    # Process a sequence
    h = np.zeros(32)
    c = np.zeros(32)
    
    gate_history = {'i': [], 'f': [], 'o': [], 'g': []}
    h_history = []
    c_history = []
    
    for t in range(seq_len):
        x = np.random.randn(10)
        h, c, gates = lstm.forward(x, h, c)
        
        for name in gate_history:
            gate_history[name].append(gates[name].mean())
        h_history.append(h.mean())
        c_history.append(c.mean())
    
    # Plot
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    ax1 = axes[0]
    for name, values in gate_history.items():
        ax1.plot(values, label=f'{name} gate')
    ax1.set_xlabel('Time step')
    ax1.set_ylabel('Mean activation')
    ax1.set_title('LSTM Gate Activations Over Time')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    ax2 = axes[1]
    ax2.plot(h_history, label='Hidden state')
    ax2.plot(c_history, label='Cell state')
    ax2.set_xlabel('Time step')
    ax2.set_ylabel('Mean value')
    ax2.set_title('LSTM State Evolution')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

visualize_lstm_gates()

## 4.4 Character-Level Language Model

In [ ]:
# Simple character-level tokenizer
text = "hello world. this is a simple text for our language model."

chars = sorted(list(set(text)))
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}

print(f"Vocabulary: {chars}")
print(f"Vocabulary size: {len(chars)}")

# Encode text
encoded = [char_to_idx[c] for c in text]
print(f"\nEncoded 'hello': {[char_to_idx[c] for c in 'hello']}")

## Summary

In this notebook, we:
1. Built a vanilla RNN from scratch
2. Visualized the vanishing gradient problem
3. Implemented an LSTM cell with gates
4. Analyzed gate activations over time
5. Set up a character-level language model

**Next:** Module 5 covers Attention and Transformers.